# Find users by name

## Qué hace

Busca los usuarios que se llaman exactamente como el nombre escrito y los muestra ordenados por id.

## Qué pasa en la base de datos

Solo lee la tabla `users`. No cambia nada.

## Qué comando usa

`src/clients/terminal/find-user-by-name.txt`. El bloque de abajo llama a la misma función, `findUserByName()`.

## Anotaciones

- El nombre está en la línea `findUserByName(...)`. Cámbialo ahí para buscar otro.
- La coincidencia es exacta y distingue mayúsculas: `Ana` no encuentra a `ana` ni a `Ana María`. Los espacios del principio y del final no cuentan.
- Si no hay ninguno, se ven los encabezados y el texto `No users with that name.`
- Si el nombre está vacío, se ve `Error: Name is required.` y no sale la tabla.
- Pulsa ejecutar en el bloque de abajo. No hace falta escribir código.

In [1]:
import os
import subprocess
from pathlib import Path

# Sube de carpeta hasta encontrar la app (la carpeta que contiene logic/, que es src/).
app = Path.cwd()
while app != app.parent and not (app / "logic").is_dir():
    app = app.parent
if not (app / "logic").is_dir():
    raise SystemExit("App not found.")

code = r"""
import Table from "cli-table3";
import { findUserByName } from "./logic/find-user-by-name.js";

try {
  const users = findUserByName("Ana");

  const table = new Table({
    head: ["id", "name", "email", "createdAt", "updatedAt"],
    style: { head: ["cyan", "bold"] },
  });
  for (const user of users) {
    table.push([user.id, user.name, user.email, user.createdAt, user.updatedAt]);
  }
  console.log(table.toString());

  if (users.length === 0) {
    console.log("No users with that name.");
  }
} catch (error) {
  console.error(`Error: ${error.message}`);
  process.exit(1);
}
"""

# Fuerza los colores del encabezado aunque la salida no sea una terminal.
env = os.environ.copy()
env["FORCE_COLOR"] = "1"
env.pop("NO_COLOR", None)

# La salida se lee en UTF-8 para que los bordes de la tabla no fallen en Windows.
completed = subprocess.run(
    ["node", "--no-warnings", "--input-type=module", "-e", code],
    cwd=app,
    capture_output=True,
    text=True,
    encoding="utf-8",
    env=env,
)
print(completed.stdout, end="")
if completed.returncode != 0:
    print(completed.stderr, end="")

┌────┬──────┬──────────────┬─────────────────────┬─────────────────────┐
│ id │ name │ email        │ createdAt           │ updatedAt           │
├────┼──────┼──────────────┼─────────────────────┼─────────────────────┤
│ 2  │ Ana  │ ana@mail.com │ 2026-10-19 14:03:57 │ 2026-10-19 14:03:57 │
└────┴──────┴──────────────┴─────────────────────┴─────────────────────┘
